In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

In [2]:
matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)
 
# Safely convert stringified lists/dicts back to Python objects
def safe_literal_eval(df, col):
    df[col] = df[col].apply(lambda x: ast.literal_eval(x))
    return df

for col in ["s2m_values", "lake_reach_ids", "lake_pld_ids"]:
    matchups = safe_literal_eval(matchups, col)

In [3]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

In [11]:
status = store.get_processing_status(source='swot-river')
if status.empty:
    processed_basins = []
else:
    processed_basins = list(status.index.get_level_values('subbasin'))
to_process = matchups[~matchups.index.isin(processed_basins)]
to_process

,outlet,outlet_id,total_area,unitarea,reservoir,custom,reach_id,sword_area,sword_distance,lake_reach_ids,...,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider,hybas_area_diff,geometry
comid,,,,,,,,,,,,,,,,,,,,,
21000001,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,398.5,152.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.034201,"MULTIPOLYGON (((5.93875 47.99125, 5.93708 47.9..."
21000012,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,324.8,194.6,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.053341,"MULTIPOLYGON (((5.79042 48.01125, 5.79042 48.0..."
21000019,POINT (5.684999999999993 47.53416666666667),EAUF-V7200010,5173.0,242.5,False,False,2.160280e+10,4530.071328,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.007977,"MULTIPOLYGON (((5.76042 47.56125, 5.76042 47.5..."
21000021,POINT (5.76916666666666 47.58083333333334),EAUF-V7200010,4758.7,8.4,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.003021,"MULTIPOLYGON (((5.80625 47.60125, 5.80708 47.6..."
21000022,POINT (5.80416666666666 47.57),EAUF-V7200010,4506.6,68.9,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.009101,"MULTIPOLYGON (((5.85875 47.57458, 5.85875 47.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,NaN,NaN,NaN,[],...,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs,-0.002093,"MULTIPOLYGON (((-112.21458 40.96042, -112.2154..."
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,NaN,NaN,NaN,[],...,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs,-0.091982,"MULTIPOLYGON (((-66.49208 18.29625, -66.49208 ..."
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,NaN,NaN,NaN,[],...,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs,0.167805,"MULTIPOLYGON (((-66.50125 18.33125, -66.50125 ..."


In [4]:
import pyarrow.dataset as ds
import itertools 

lake_ids = list(itertools.chain.from_iterable(matchups['lake_reach_ids']))
river_reaches = matchups['reach_id'].dropna().astype(int).to_list()
all_reaches = lake_ids + river_reaches

fields=[
    'reach_id', 'time','wse', 'wse_u', 'wse_r_u',
    'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u',
    'width', 'width_u',
    'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse',
    'layovr_val', 'node_dist', 'loc_offset', 'xtrk_dist',
    'reach_q', 'reach_q_b', 'dark_frac', 'ice_clim_f', 'partial_f',
    'obs_frac_n', 'xovr_cal_q', 'dry_trop_c', 'wet_trop_c', 'iono_c', 'xovr_cal_c'
]

reach_dir  = datasets / 'hydrocron' / 'reach'
list(reach_dir.glob('*.parquet'))

swot = []
for reach_file in reach_dir.glob('*_hydrocron_reach.parquet'):
    dataset = ds.dataset(reach_file, format="parquet")
    table = dataset.to_table(
        columns=fields,
        filter=(ds.field("reach_id").isin(all_reaches))
    )
    swot.append(table.to_pandas())
    
all_swot = pd.concat(swot)
all_swot = all_swot[all_swot['wse'] != -999999999999.0]
all_swot['d_wse'] = all_swot['wse'] - all_swot.groupby('reach_id')['wse'].transform('median')
all_swot['d_width'] = all_swot['width'] - all_swot.groupby('reach_id')['width'].transform('median')

all_swot = all_swot.set_index(['reach_id','time'])

In [ ]:
# from deltalake import DeltaTable
# dt = DeltaTable(store.metadata_uri)
# dt.delete(predicate=f"source='swot-river'")


In [5]:
# status = store.get_processing_status(source='swot-river')
# if status.empty:
#     processed_basins = []
# else:
#     processed_basins = list(status.index.get_level_values('subbasin'))
# to_process = matchups[~matchups.index.isin(processed_basins)]

to_process = matchups

def get_best_record(grp):
    return grp.sort_values('wse_u').iloc[0]

    
def get_swot_r(reach_id):
    if np.isnan(reach_id):
        return None
        
    if reach_id in all_swot.index.get_level_values('reach_id'):
        swot = all_swot.xs(reach_id, level='reach_id')
        swot.index = pd.to_datetime(swot.index).tz_localize('UTC')

        if any(swot.index.duplicated()):
            swot = swot.groupby(swot.index.name).apply(get_best_record)

        return swot.add_suffix('_river')


max_batch_size = 256
b = False
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        reach_id = row['reach_id']
        subbasin_data[comid] = get_swot_r(reach_id)

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'swot-river', subbasin_data, mode='append')
            subbasin_data = {}

    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'swot-river', subbasin_data, mode='append')

100%|██████████| 643/643 [13:21<00:00,  1.25s/it] 


In [ ]:


df.groupby('time').apply(get_best_record)

In [ ]:
grp.sort_values('wse_u_river').iloc[0]

In [ ]:
df.index.duplicated()

In [ ]:
processed_basins = store.get_processing_status(source='swot-river')
processed_basins[processed_basins['has_data']]

In [ ]:
processed_basins[processed_basins['has_data']]['basin'].value_counts()

In [ ]:
df = store.read_dynamic('USGS-15908000', subbasin=['USGS-15908000'])
df

In [ ]:
df['glow-s'].set_index(["subbasin", "date"])

In [ ]:
store.compact()

In [ ]:
matchups['lake_pld_ids'].apply(len).gt(0).sum()

In [ ]:
# pld_fields = [
#     "n_overlap", "wse", "wse_u", "wse_r_u", "wse_std",
#     "area_total", "area_tot_u", "area_detct", "area_det_u",
#     "dark_frac", "xovr_cal_q",  "layovr_val", "xtrk_dist",
#     "quality_f",  "partial_f", "ice_clim_f",
#     "dry_trop_c", "wet_trop_c", "iono_c", "xovr_cal_c"
# ]



# def get_swot_l(pld_ids: list):
#     lake_dfs = []
#     for pld_id in pld_ids:
#         path = Path(swot_lake_dir / f"{pld_id}.parquet")
#         if path.is_file():
#             swot = pd.read_parquet(path)[pld_fields]
#             swot = swot.replace(-999999999999.0, np.nan)
#             swot.dropna(subset=['wse', 'area_total'])
    
#             wse = swot['wse']
#             area = swot['area_total']
#             swot['d_wse'] = wse - wse.median()
#             swot['d_area'] = area - area.median()
#             swot['d_volume'] = swot['d_wse'] * (0.5*(area + area.median()))
#             lake_dfs.append(swot)
#         else:
#             lake_dfs.append([])

#     df_lens = [len(d) for d in lake_dfs]
#     if any(l>0 for l in df_lens):
#         swot = lake_dfs[np.argmax(df_lens)]
#         swot.index = swot.index.normalize().tz_convert('UTC')
#         return swot
#     else:
#         new_fields = ['d_wse', 'd_area', 'd_volume']
#         return pd.DataFrame(columns = pld_fields + new_fields)
    

# def get_gauge(site_id):
#     gauge = facade.get_daily_values(site_id, t1, t2).droplevel('site_id')
#     gauge.index = gauge.index.tz_localize('UTC')
#     return gauge[['discharge']]


In [ ]:
# %%
date_range = pd.date_range(start=t1, end=t2, freq='D')
# to_process = get_reaches_to_process()
to_process = matchups.index.unique()

for hybas_id in tqdm(to_process, total=len(to_process), desc="Writing files"):
    matchups_ix = matchups.loc[hybas_id]
    
    swot_r_df = get_swot_r(matchups_ix['reach_id']).add_suffix('_river')
    zds.add_dynamic_source(hybas_id, 'swot_river', swot_r_df)

    # swot_l_df = get_swot_l(matchups_ix['lake_pld_ids']).add_suffix('_lake')
    # zds.add_dynamic_source(hybas_id, 'swot_lake', swot_l_df)
    
    glow_df = get_glow_s(matchups_ix['mb_values'])
    zds.add_dynamic_source(hybas_id, 'glow_s', glow_df)

    # if matchups_ix['custom'] and not hybas_id=='outlet':
    #     gauge_df = get_gauge(hybas_id)
    # else:
    #     gauge_df = pd.DataFrame(columns = ['discharge'])
    # gauge_df['sp_discharge'] = gauge_df['discharge'] / matchups_ix['total_area']
    # zds.add_dynamic_source(hybas_id, 'gauge', gauge_df)

In [ ]:
zds.add_dynamic_source('swot_river', swot_r_df, hybas_id)

In [ ]:
gauge_df2 = pd.DataFrame(columns = ['discharge'])
gauge_df2['sp_discharge'] = gauge_df2['discharge'] / matchups_ix['total_area']

gauge_df2

In [ ]:
(gauge_df / matchups_ix['total_area']).plot()

In [ ]:
matchups_ix['total_area']

In [ ]:
hybas_id

In [ ]:
to_process